# Processamento de Linguagem Natural e de Imagens — Aula 02
# Vetorização de textos: Bag-of-Words, TF-IDF e similaridade de cossenos

**Curso:** Processamento de Linguagem Natural e de Imagens — 2026.2
**Corpus de trabalho:** `savedrecs1.xls` — exportação da *Web of Science* com 1.000 registros bibliográficos sobre **ligas de alta entropia** (*high-entropy alloys*, HEAs).

---

### Do que trata esta aula

Na Aula 01 aprendemos a **preparar** texto: tokenização, remoção de pontuação, *stop words*, *stemming* e lematização. O resultado daquele processo é uma lista de tokens — ou seja, ainda é texto.

O problema é que nenhum algoritmo de aprendizado de máquina consome texto. Regressão logística, SVM, *k*-means, redes neurais: todos operam sobre **vetores de números reais**. Existe portanto uma ponte obrigatória entre o texto limpo e o modelo, e essa ponte se chama **vetorização** (ou *feature extraction* de texto).

Esta aula percorre as duas técnicas clássicas dessa ponte:

| Técnica | Ideia central | O que captura | O que ignora |
|---|---|---|---|
| **Bag-of-Words (BOW)** | contar quantas vezes cada palavra do vocabulário aparece no documento | presença e frequência de termos | ordem, sintaxe, sinônimos |
| **TF-IDF** | contar, mas **descontar** as palavras que aparecem em todo lugar | termos *discriminativos* de cada documento | ordem, sintaxe, sinônimos |

E a métrica que usaremos para comparar os vetores resultantes:

- **Similaridade de cossenos** — mede o *ângulo* entre dois vetores de documento, ignorando o comprimento deles.

Ao final, teremos construído um **motor de busca acadêmico** funcional: você digita uma consulta em linguagem natural (`"hydrogen storage in refractory alloys"`) e o notebook devolve, ordenados por relevância, os artigos do corpus que mais se aproximam dela — junto com título, periódico e ano.

### Objetivos de aprendizagem

Ao final desta aula, você deve ser capaz de:

1. Explicar por que um modelo de ML não pode receber texto bruto e o que exatamente a vetorização produz.
2. Construir uma **matriz documento-termo** com `CountVectorizer` e interpretar suas dimensões, sua esparsidade e seu vocabulário.
3. Acoplar um **tokenizador customizado** (spaCy) ao vetorizador do scikit-learn, controlando lematização, *stop words* e filtragem de tokens.
4. Calcular **similaridade de cossenos** entre documentos — na mão com NumPy, com SciPy e com scikit-learn — e justificar por que ela é preferível à distância euclidiana em texto.
5. Explicar a fórmula do **TF-IDF** (incluindo a variante *smooth* que o scikit-learn usa) e verificá-la numericamente.
6. Usar `min_df`, `max_df` e `ngram_range` para controlar o tamanho e a qualidade do vocabulário.
7. Implementar **recuperação de informação** (busca e recomendação de artigos) sobre um corpus científico real.
8. Reconhecer as **limitações** dessas representações (vocabulário fixo, sinonímia, polissemia, perda de ordem) — que motivarão os *word embeddings* na próxima aula.

### Créditos e origem do material

Este notebook é uma adaptação, em português e com material expandido, do notebook de vetorização do curso aberto **NLP Demystified**:

- https://nlpdemystified.org
- https://github.com/futuremojo/nlp-demystified

A estrutura conceitual (BOW → similaridade → *n*-gramas → TF-IDF → busca) segue a do curso original. Todos os textos explicativos foram reescritos e ampliados para esta disciplina, e **todos os exemplos foram refeitos sobre o corpus `savedrecs1.xls`** — o mesmo arquivo usado nas Aulas 01 e 03 — em vez dos corpora de brinquedo e do dataset *20 newsgroups* do original. A vantagem é dupla: o corpus é real, é científico e é o mesmo do trabalho da disciplina.

---
# 0. Preparação do ambiente

Precisamos de cinco bibliotecas:

| Biblioteca | Papel nesta aula |
|---|---|
| **pandas** | ler o `.xls` e manipular a tabela bibliográfica |
| **xlrd** | *engine* que o pandas usa para abrir arquivos `.xls` antigos (formato binário do Excel 97-2003, que é o exportado pela Web of Science) |
| **spaCy** | tokenização, lematização e *stop words* de qualidade linguística |
| **scikit-learn** | `CountVectorizer`, `TfidfVectorizer`, similaridade de cossenos, *k*-means |
| **SciPy / NumPy** | matrizes esparsas e álgebra linear |

**Se estiver no Google Colab:** execute as duas células abaixo. O Colab reinicia o ambiente após períodos de inatividade — se isso acontecer, será preciso reinstalar. Lembre-se também de fazer *upload* do arquivo `savedrecs1.xls` para a sessão (ícone de pasta na barra lateral esquerda).

**Se estiver rodando localmente (Jupyter/VS Code):** basta executar uma vez; nas próximas aulas o ambiente já estará pronto.

In [ ]:
# Instalação das bibliotecas. Em ambiente local, execute apenas uma vez.
!pip install -q spacy scikit-learn pandas xlrd matplotlib

In [ ]:
# O spaCy separa o "código" (a biblioteca) dos "modelos estatísticos" (os pesos treinados).
# en_core_web_sm é o menor modelo de inglês: rápido, leve e suficiente para tokenizar e lematizar.
# Nosso corpus é de artigos em inglês, por isso o modelo em inglês.
!python -m spacy download en_core_web_sm
!python -m spacy info

### Importações

Repare no que estamos importando e por quê:

- `CountVectorizer` — produz a matriz de **contagens** (BOW bruto ou binário).
- `TfidfVectorizer` — produz a matriz **TF-IDF**. É, na prática, um `CountVectorizer` seguido de uma reponderação.
- `cosine_similarity` — versão vetorizada (calcula todos os pares de uma vez) da similaridade de cossenos.
- `scipy.spatial` — traz a *distância* de cossenos, útil para comparar dois vetores isolados.

In [ ]:
import numpy as np
import pandas as pd
import spacy

from scipy import spatial
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

# Deixa a exibição de DataFrames mais confortável para uso em sala.
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 160)

print('numpy  ', np.__version__)
print('pandas ', pd.__version__)
print('spacy  ', spacy.__version__)

---
# 1. O corpus: uma exportação da Web of Science

## 1.1 O que é o `savedrecs1.xls`

A **Web of Science** (Clarivate) é uma base de dados bibliográficos indexada. Quando você faz uma busca lá — neste caso, algo próximo de *"high entropy alloys"* — pode exportar os resultados em lotes de até 1.000 registros. O arquivo `savedrecs1.xls` é exatamente isso: **um lote de 1.000 artigos científicos** sobre ligas de alta entropia, com 70 colunas de metadados cada.

Ligas de alta entropia (HEAs) são uma classe de materiais metálicos relativamente recente: em vez de um elemento principal com pequenas adições (como o aço, que é ferro com pouco carbono), elas misturam cinco ou mais elementos em proporções quase equimolares. Isso gera alta entropia configuracional e propriedades incomuns — dureza, resistência a corrosão, estabilidade em alta temperatura. É um tema *quente* na ciência dos materiais, o que explica a quantidade de publicações recentes no arquivo.

Do ponto de vista de PLN, o que nos interessa é que temos **1.000 documentos de texto técnico, em inglês, de domínio restrito e escritos por especialistas**. Isso é ótimo para ensinar vetorização, porque:

- o vocabulário é especializado e repetitivo → dá para *ver* o efeito do IDF;
- os documentos são naturalmente agrupáveis por subtema (corrosão, deformação, revestimentos, filmes finos...) → dá para *ver* a similaridade funcionando;
- há metadados (título, periódico, ano, palavras-chave) para validar qualitativamente os resultados da busca.

> **Nota sobre o formato.** O `.xls` da Web of Science é o formato binário antigo do Excel. O pandas precisa da *engine* `xlrd` para lê-lo. Se você receber `ImportError: Missing optional dependency 'xlrd'`, rode `!pip install xlrd`.

In [ ]:
# Leitura do arquivo exportado da Web of Science.
# Se o arquivo estiver em outra pasta, ajuste o caminho abaixo.
CAMINHO = 'savedrecs1.xls'

dados = pd.read_excel(CAMINHO)

print(f'Registros (linhas)  : {dados.shape[0]}')
print(f'Metadados (colunas) : {dados.shape[1]}')

In [ ]:
# Todas as 70 colunas disponíveis na exportação.
list(dados.columns)

## 1.2 Selecionando o que interessa

Das 70 colunas, cinco bastam para esta aula:

| Coluna | Uso |
|---|---|
| `Abstract` | **é o nosso documento** — o texto que será vetorizado |
| `Article Title` | rótulo legível para conferir os resultados da busca |
| `Author Keywords` | palavras-chave dadas pelos autores; servirão de "gabarito" informal |
| `Source Title` | periódico onde saiu o artigo |
| `Publication Year` | ano de publicação |

Também descartamos registros sem resumo — um documento vazio não contribui com nada para o vocabulário e ainda estraga o cálculo de similaridade (vetor nulo → cosseno indefinido, o scikit-learn devolve 0).

In [ ]:
COLUNAS = ['Article Title', 'Abstract', 'Author Keywords', 'Source Title', 'Publication Year']

df = (dados[COLUNAS]
      .dropna(subset=['Abstract'])      # sem resumo, sem documento
      .reset_index(drop=True))          # reindexa de 0..N-1 (importante: os índices
                                        # do DataFrame terão que casar com as linhas
                                        # da matriz documento-termo)

print(f'Documentos utilizáveis: {len(df)}')
df.head(5)

## 1.3 Conhecendo o texto antes de vetorizar

Uma regra que vale para qualquer projeto de PLN: **leia o seu corpus antes de processá-lo.** Decisões como "devo remover números?", "devo lematizar?", "*stop words* atrapalham?" só fazem sentido depois de olhar para o texto real.

Vamos inspecionar um documento inteiro.

In [ ]:
i = 0
print('TÍTULO   :', df['Article Title'][i])
print('PERIÓDICO:', df['Source Title'][i])
print('ANO      :', int(df['Publication Year'][i]))
print('KEYWORDS :', df['Author Keywords'][i])
print()
print('RESUMO:')
print(df['Abstract'][i])

Observe as características típicas de texto científico, que terão consequências diretas na vetorização:

- **Fórmulas químicas como tokens**: `CoCrFeMnNi`, `Al0.25CoCrFeNi`, `TiZrHfScMo`. São palavras únicas, altamente informativas, e serão destruídas se você aplicar *lowercase* de forma ingênua ou filtrar por comprimento.
- **Números com unidade**: `1200 degrees C`, `750 MPa`. Em geral removemos números, pois `1200` sozinho não diz muito para busca temática.
- **Siglas**: `HEA`, `BCC`, `FCC`, `DFT`, `XRD`. Muito informativas — não devem ser filtradas.
- **Hífens e parênteses** em abundância: `high-entropy`, `(FeCrNiCo)AlxCuy`. A tokenização precisa ser decidida com cuidado.
- **Variação morfológica**: `alloy` / `alloys`, `deform` / `deformed` / `deformation`. Sem lematização, isso vira colunas *diferentes* na matriz — o modelo não sabe que são a mesma coisa.

In [ ]:
# Estatísticas de tamanho dos documentos.
tam_car = df['Abstract'].str.len()
tam_pal = df['Abstract'].str.split().str.len()

resumo = pd.DataFrame({'caracteres': tam_car.describe(), 'palavras': tam_pal.describe()})
print(resumo.round(1))

In [ ]:
# Distribuição temporal: o corpus reflete o crescimento explosivo do tema.
print('Publicações por ano:')
print(df['Publication Year'].value_counts().sort_index().to_string())

In [ ]:
# Onde esses artigos são publicados (os 10 periódicos mais frequentes).
df['Source Title'].value_counts().head(10)

## 1.4 Definindo o corpus e fixando o vocabulário conceitual

Antes de escrever a primeira linha de vetorização, fixemos os termos que serão usados o tempo todo:

| Termo | Definição | No nosso caso |
|---|---|---|
| **Documento** | a unidade de texto que será representada por um vetor | um *abstract* |
| **Corpus** | a coleção de documentos | os 1.000 *abstracts* |
| **Token** | uma ocorrência individual de palavra no texto | cada palavra de cada resumo |
| **Tipo** (*type*) | uma palavra distinta, independentemente de quantas vezes ocorre | entrada do vocabulário |
| **Vocabulário** | o conjunto de todos os tipos do corpus, com um índice inteiro para cada | ~5.000 termos, dependendo do pré-processamento |
| **Matriz documento-termo** | matriz com uma linha por documento e uma coluna por termo do vocabulário | 1.000 × ~5.000 |

Um detalhe que causa confusão: **o vocabulário é definido pelo corpus, não pela língua**. Se a palavra `titanium` não aparece em nenhum resumo, ela simplesmente não existe para o modelo — e um documento novo que a contenha terá essa informação descartada. É o problema das **palavras fora do vocabulário** (*out-of-vocabulary*, OOV), que voltará a aparecer mais adiante.

In [ ]:
# O corpus, finalmente: uma lista de strings.
corpus = df['Abstract'].tolist()

print(f'Tipo    : {type(corpus)}')
print(f'Tamanho : {len(corpus)} documentos')
print()
print('Primeiros 300 caracteres do documento 0:')
print(corpus[0][:300], '...')

---
# 2. Bag-of-Words (saco de palavras)

## 2.1 A ideia

*Bag-of-words* é, provavelmente, a representação mais simples que ainda funciona. O procedimento tem três passos:

1. **Construir o vocabulário**: percorra todo o corpus, colete todas as palavras distintas e atribua a cada uma um índice inteiro (0, 1, 2, ...). Se o corpus tem *V* palavras distintas, o vocabulário tem tamanho *V*.
2. **Representar cada documento** por um vetor de tamanho *V*, onde a posição *j* guarda quantas vezes o termo *j* aparece naquele documento.
3. **Empilhar** os vetores: obtemos a **matriz documento-termo**, de dimensão (nº de documentos) × *V*.

O nome vem da metáfora: é como se cada documento fosse esvaziado dentro de um saco e as palavras se embaralhassem. Sobra quem está lá e quantas vezes — some a informação de **em que ordem**.

### Um exemplo minúsculo

Com o corpus `["gato preto", "gato branco gato"]`, o vocabulário é `{branco: 0, gato: 1, preto: 2}` e a matriz é:

| | branco | gato | preto |
|---|---|---|---|
| **doc 0** | 0 | 1 | 1 |
| **doc 1** | 1 | 2 | 0 |

### O que se ganha e o que se perde

**Ganha-se:** simplicidade, velocidade, interpretabilidade total (cada coluna é uma palavra que você pode ler) e desempenho surpreendentemente bom em classificação de tópicos, filtragem de spam e recuperação de informação.

**Perde-se:**

- **Ordem e sintaxe.** "O revestimento protege o substrato" e "O substrato protege o revestimento" têm vetores *idênticos*. Negação então é fatal: "not corrosion resistant" e "corrosion resistant" ficam quase iguais.
- **Semântica.** `alloy` e `metallic material` são colunas distintas e ortogonais entre si — o modelo não faz ideia de que são relacionadas.
- **Compacidade.** O vetor tem o tamanho do vocabulário inteiro (milhares de posições), mas um resumo usa umas 100 palavras distintas. Ou seja: **mais de 98% das entradas são zero**. Isso se chama *esparsidade*, e é por isso que a matriz precisa ser guardada em formato comprimido.

Variações comuns do BOW:

- **Frequência bruta** — a contagem, como no exemplo. Padrão do `CountVectorizer`.
- **Binário** — apenas 0/1 marcando presença. Útil quando repetir a palavra não deve valer mais, e para documentos curtos.
- **TF-IDF** — contagens reponderadas pela raridade do termo no corpus. É a Seção 3.

## 2.2 Um mini-corpus para enxergar a matriz

Uma matriz 1.000 × 5.000 não cabe na tela e não ensina nada visualmente. Então começaremos com um recorte didático: **cinco títulos reais do nosso arquivo**, escolhidos por serem curtos e por tratarem de subtemas diferentes (refratárias, armazenamento de hidrogênio, fadiga, corrosão, magnetismo).

Repare: continuamos usando o `savedrecs1.xls`. Não é um corpus de brinquedo inventado — são artigos de verdade, apenas em pedaço pequeno o bastante para caber numa tabela impressa. Depois de entender a mecânica aqui, voltaremos aos 1.000 resumos completos.

In [ ]:
# Cinco títulos reais do corpus, de subtemas distintos.
INDICES_MINI = [0, 7, 33, 53, 67]

mini_corpus = df['Article Title'].iloc[INDICES_MINI].tolist()

for k, t in enumerate(mini_corpus):
    print(f'doc{k}: {t}')

## 2.3 `CountVectorizer`: a matriz documento-termo em uma linha

Poderíamos implementar BOW do zero com um dicionário e algumas listas — e é um bom exercício de programação. Mas o scikit-learn já traz uma implementação eficiente e cheia de opções úteis:

- Documentação: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html

O `CountVectorizer` segue a API padrão do scikit-learn, baseada em dois verbos que você verá o curso inteiro:

- **`fit(corpus)`** — *aprende* alguma coisa a partir dos dados. Aqui, aprende o vocabulário: quais palavras existem e qual índice cada uma recebe.
- **`transform(corpus)`** — *aplica* o que foi aprendido, convertendo textos em vetores.
- **`fit_transform(corpus)`** — faz os dois de uma vez, que é o caso comum no corpus de treino.

Essa separação é fundamental e voltará em toda a disciplina. A regra é: **`fit` só no conjunto de treino; `transform` em tudo o mais** (validação, teste, consulta do usuário). Se você chamar `fit_transform` numa consulta nova, vai criar um vocabulário novo e incompatível com a matriz do corpus — e a comparação perde o sentido.

In [ ]:
# Instanciação com as opções padrão.
vectorizer = CountVectorizer()

# fit  -> aprende o vocabulário a partir do mini-corpus
# transform -> devolve a matriz de contagens
bow = vectorizer.fit_transform(mini_corpus)

print(f'Dimensões da matriz: {bow.shape}   (documentos × termos)')

### O vocabulário aprendido

Duas formas de inspecioná-lo:

- `get_feature_names_out()` — devolve os termos **em ordem de coluna** (é o rótulo do eixo horizontal da matriz).
- `vocabulary_` — devolve o dicionário `{termo: índice_da_coluna}`.

Note que o `CountVectorizer` já fez, sozinho, várias etapas de pré-processamento que na Aula 01 fizemos manualmente: tokenizou por uma expressão regular, converteu tudo para minúsculas e descartou a pontuação e os tokens de um só caractere.

In [ ]:
termos = vectorizer.get_feature_names_out()

print(f'Tamanho do vocabulário: {len(termos)}')
print()
print(termos)

In [ ]:
# O mapeamento termo -> índice da coluna.
vectorizer.vocabulary_

Repare em dois detalhes reveladores do vocabulário acima:

1. **`alloy` e `alloys` são entradas separadas.** Para o `CountVectorizer` padrão são palavras completamente distintas — colunas diferentes, sem qualquer relação. Isso desperdiça informação: dois documentos que falam da mesma coisa aparentam ter menos em comum do que realmente têm. É exatamente o problema que a **lematização** resolve, e faremos isso na Seção 2.6.
2. **As fórmulas químicas sobreviveram**, mas em minúsculas: `crmnfeconi`, `tizrhfscmo`, `mncr2o4`. Continuam sendo tokens únicos e muito informativos — mas perdemos a capitalização que, em química, carrega significado (`Co` = cobalto, `CO` = monóxido de carbono). É um argumento concreto para desligar o *lowercase* em corpora técnicos.

## 2.4 Matrizes esparsas: por que a saída não é um array comum

Olhe o tipo do objeto devolvido: não é `numpy.ndarray`, e sim uma **matriz esparsa comprimida** (`scipy.sparse.csr_matrix`).

O motivo é econômico. Nosso corpus completo produzirá uma matriz de 1.000 × ~5.000 = 5 milhões de células, das quais cerca de 98,8% são zero. Guardar 5 milhões de zeros em `float64` custaria 40 MB de memória para representar ~60 mil números úteis. O formato **CSR** (*Compressed Sparse Row*) guarda apenas as posições não nulas: para cada valor, a linha e a coluna onde ele está.

E, além de economizar memória, o formato esparso é *mais rápido*: multiplicações de matriz saltam os zeros em vez de multiplicá-los.

- Documentação: https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_matrix.html

In [ ]:
print('Tipo do objeto :', type(bow))
print()
print('Formato        :', bow.shape)
print('Células totais :', bow.shape[0] * bow.shape[1])
print('Não nulas (nnz):', bow.nnz)
print(f'Densidade      : {100 * bow.nnz / (bow.shape[0] * bow.shape[1]):.2f}%')

In [ ]:
# Imprimindo a matriz esparsa: cada linha é "(documento, termo)  contagem".
# Só aparecem as células não nulas.
print(bow)

### A versão densa

Para inspeção visual, `toarray()` expande a matriz esparsa num array NumPy comum. **Cuidado:** faça isso apenas em recortes pequenos — expandir a matriz completa de um corpus grande pode estourar a memória.

Vamos montar um `DataFrame` do pandas com os termos como nomes de coluna. É a mesma tabela dos slides, agora com dados reais.

In [ ]:
dtm = pd.DataFrame(
    bow.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f'doc{k}' for k in range(len(mini_corpus))]
)

# Exibindo só as 14 primeiras colunas para caber na tela.
dtm.iloc[:, :14]

**Como ler essa tabela.** Cada **linha** é um documento inteiro reduzido a um vetor de contagens; cada **coluna** é um termo do vocabulário. A linha `doc3`, por exemplo, tem 1 em `corrosion` e 1 em `crmnfeconi`, e zero em `hydrogen` — porque aquele artigo trata de corrosão por pite na liga CrMnFeCoNi.

Note também que a coluna `high` e a coluna `entropy` estão preenchidas com 1 em **todas** as linhas. Elas ocupam espaço, custam processamento e **não distinguem nada** — todo documento do corpus fala de *high entropy alloys*. Guarde essa observação: ela é a motivação exata do TF-IDF, na Seção 3.

In [ ]:
# Indexação e fatiamento funcionam normalmente na matriz esparsa.
print('Documento 0 (esparso):')
print(bow[0])
print()
print('Documentos 0 e 1, densos:')
print(bow[0:2].toarray())

## 2.5 Tokenizador customizado com spaCy + BOW binário

O tokenizador padrão do `CountVectorizer` é uma expressão regular (`token_pattern=r"(?u)\b\w\w+\b"`: sequências de dois ou mais caracteres alfanuméricos). É rápido, mas cego a linguística: não sabe lematizar, não conhece *stop words*, não distingue substantivo de verbo.

Como vimos na Aula 01, o spaCy faz tudo isso. O `CountVectorizer` aceita um **callback** no parâmetro `tokenizer`: para cada documento, ele chama a sua função e espera receber de volta uma lista de tokens. Isso nos dá controle total sobre o pré-processamento.

Vamos aproveitar para mudar outra coisa: o parâmetro `binary=True` faz a matriz guardar apenas **0 ou 1** (presença/ausência) em vez da contagem. Para títulos e resumos curtos isso costuma ser razoável — repetir uma palavra três vezes num resumo não a torna três vezes mais relevante.

> ⚠️ **Detalhe de versão.** A partir do scikit-learn 1.2, passar `tokenizer=` sem também passar `token_pattern=None` gera o aviso *"The parameter 'token_pattern' will not be used since 'tokenizer' is not None"*. Não é um erro, mas passe `token_pattern=None` explicitamente para deixar a intenção clara e o *output* limpo.

In [ ]:
# Carregamos o modelo estatístico de inglês, como na Aula 01.
nlp = spacy.load('en_core_web_sm')

# Callback simples: tokeniza com o spaCy e devolve o texto dos tokens,
# descartando pontuação.
def spacy_tokenizer(doc):
    return [t.text for t in nlp(doc) if not t.is_punct]

# Teste rápido do callback antes de plugá-lo no vetorizador —
# sempre teste o tokenizador isoladamente, é onde moram os bugs silenciosos.
print(spacy_tokenizer(mini_corpus[1]))

In [ ]:
vectorizer = CountVectorizer(
    tokenizer=spacy_tokenizer,   # usa o spaCy no lugar da regex padrão
    token_pattern=None,          # explicita que a regex não será usada
    lowercase=False,             # preserva a capitalização das fórmulas químicas
    binary=True                  # presença/ausência em vez de contagem
)

bow_bin = vectorizer.fit_transform(mini_corpus)

print(f'Dimensões: {bow_bin.shape}')
print()
print(vectorizer.get_feature_names_out())

Compare com o vocabulário da Seção 2.3 e observe três mudanças:

1. `TiZrHfScMo` e `MnCr2O4` preservaram a grafia original — era o que queríamos ao desligar o *lowercase*.
2. Tokens curtos como `A`, `of`, `on`, `in`, `and` — antes descartados pela regex de dois caracteres — passaram a existir. Ganhamos ruído.
3. **Pior: o vocabulário agora tem `Alloy` e `alloy`, `Alloys` e `alloys`, `High` e `high`, `Entropy` e `entropy` como entradas separadas.** A mesma palavra ocupa duas colunas, só porque um título usa caixa alta e o outro não.

Ou seja: **trocamos um problema por outro**. Ganhamos fidelidade química e perdemos tanto a filtragem de palavras irrelevantes quanto a unificação por caixa. O item 3 terá uma consequência dramática — e bastante instrutiva — na próxima seção. Guarde-o.

## 2.6 Comparando estratégias de pré-processamento

Não existe "o" pré-processamento correto — existe o pré-processamento adequado à tarefa. Para deixar isso concreto, vamos construir **quatro tokenizadores** e medir o efeito de cada um sobre o tamanho do vocabulário, usando o corpus completo de 1.000 resumos:

1. **regex padrão** — o comportamento de fábrica do `CountVectorizer`;
2. **spaCy, texto puro** — só remove pontuação e espaços;
3. **spaCy, apenas alfabéticos** — descarta números e tokens mistos como `750`, `2.5`;
4. **spaCy, lema + sem *stop words*** — o mais agressivo, e o que usaremos no TF-IDF.

Cada filtro adicional encolhe o vocabulário. Um vocabulário menor significa menos colunas, menos memória, menos ruído — mas também risco de jogar fora informação útil. O ponto de equilíbrio depende do que você vai fazer com os vetores.

In [ ]:
# Desligamos os componentes do pipeline que não usamos aqui.
# 'ner' (entidades nomeadas) e 'parser' (análise sintática) são caros;
# 'tagger' precisa ficar ligado, porque o lematizador depende da classe gramatical.
componentes_desnecessarios = ['ner', 'parser']

def tok_texto(doc):
    """spaCy, texto puro: só descarta pontuação e espaços."""
    with nlp.disable_pipes(*componentes_desnecessarios):
        return [t.text.lower() for t in nlp(doc) if not t.is_punct and not t.is_space]

def tok_alpha(doc):
    """spaCy, apenas tokens alfabéticos (descarta números e códigos mistos)."""
    with nlp.disable_pipes(*componentes_desnecessarios):
        return [t.text.lower() for t in nlp(doc)
                if not t.is_punct and not t.is_space and t.is_alpha]

def tok_lema(doc):
    """spaCy, lema + alfabéticos + sem stop words. O mais agressivo."""
    with nlp.disable_pipes(*componentes_desnecessarios):
        return [t.lemma_.lower() for t in nlp(doc)
                if not t.is_punct and not t.is_space and t.is_alpha and not t.is_stop]

# Demonstração em um trecho, para ver a diferença lado a lado.
trecho = corpus[0][:180]
print('TRECHO ORIGINAL:'); print(trecho); print()
print('texto :', tok_texto(trecho)[:18]); print()
print('alpha :', tok_alpha(trecho)[:18]); print()
print('lema  :', tok_lema(trecho)[:18])

Compare as três últimas linhas. Na versão `lema`, `attracted` virou `attract`, `alloys` virou `alloy`, e artigos, preposições e verbos auxiliares (`have`, `to`, `their`, `due`) desapareceram. O que sobra é praticamente a lista de conceitos do resumo — que é justamente o que queremos para busca temática.

Agora a comparação quantitativa sobre os 1.000 documentos. A célula abaixo demora cerca de meio minuto: o spaCy precisa processar todo o corpus quatro vezes.

In [ ]:
%%time
comparacao = []

configuracoes = [
    ('1. regex padrão (sklearn)', dict()),
    ('2. spaCy, texto puro',      dict(tokenizer=tok_texto,  token_pattern=None, lowercase=False)),
    ('3. spaCy, só alfabéticos',  dict(tokenizer=tok_alpha,  token_pattern=None, lowercase=False)),
    ('4. spaCy, lema s/ stopword',dict(tokenizer=tok_lema,   token_pattern=None, lowercase=False)),
]

for nome, kwargs in configuracoes:
    v = CountVectorizer(**kwargs)
    m = v.fit_transform(corpus)
    comparacao.append({
        'estratégia'  : nome,
        'vocabulário' : m.shape[1],
        'células'     : m.shape[0] * m.shape[1],
        'não nulas'   : m.nnz,
        'densidade %' : round(100 * m.nnz / (m.shape[0] * m.shape[1]), 2),
    })

pd.DataFrame(comparacao)

Três leituras importantes dessa tabela:

- **A lematização é o filtro mais eficaz** para reduzir o vocabulário sem perder conteúdo: ela funde `alloy`/`alloys`, `deform`/`deformed`/`deforming` em uma única coluna. Menos colunas com *o mesmo* conteúdo semântico é ganho puro.
- **A densidade fica em torno de 1%**. Ou seja, 99 de cada 100 células da matriz são zero. Isso não é um defeito do nosso corpus — é a natureza do BOW e a razão de existirem as matrizes esparsas.
- **Remover números encolhe bastante o vocabulário**, porque cada valor numérico distinto (`750`, `1200`, `0.25`) vira um tipo próprio que aparece em pouquíssimos documentos. Em busca temática eles só atrapalham. Em outra tarefa — extrair propriedades mecânicas dos resumos, por exemplo — seriam justamente a informação mais valiosa. **Depende da tarefa.**

---
## 2.7 Similaridade de cossenos

Agora que cada documento é um vetor, "quão parecidos são dois documentos?" vira uma pergunta de geometria. A resposta padrão em PLN é a **similaridade de cossenos**: o cosseno do ângulo entre os dois vetores.

$$\text{sim}(\mathbf{a}, \mathbf{b}) \;=\; \cos(\theta) \;=\; \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|} \;=\; \frac{\sum_{i} a_i b_i}{\sqrt{\sum_i a_i^2}\,\sqrt{\sum_i b_i^2}}$$

O numerador é o **produto escalar**: soma dos produtos termo a termo. Como os vetores BOW nunca têm valores negativos, o produto escalar só cresce quando os *mesmos* termos aparecem nos *dois* documentos. O denominador **normaliza** pelos comprimentos, tirando o efeito do tamanho.

Interpretação dos valores (para vetores não negativos, o intervalo é [0, 1]):

| Valor | Significado |
|---|---|
| **1,0** | mesma direção — as mesmas palavras nas mesmas proporções |
| **~0,5** | sobreposição parcial de vocabulário |
| **0,0** | vetores ortogonais — **nenhuma** palavra em comum |

### Por que cosseno e não distância euclidiana?

Essa é a pergunta que sempre aparece em aula, e a resposta é o tamanho do documento. Imagine um resumo curto e sua versão estendida, que repete os mesmos conceitos com mais detalhamento. Os dois **apontam na mesma direção** no espaço de termos, mas o segundo tem um vetor bem mais **longo**. A distância euclidiana entre eles seria grande — ela os julgaria diferentes — enquanto o cosseno os reconhece como praticamente idênticos.

Como documentos de tamanhos muito distintos são a regra e não a exceção, o cosseno é a escolha natural. Vamos demonstrar isso numericamente.

In [ ]:
# Demonstração: o efeito do comprimento do documento.
a = "high entropy alloy"                                         # curto
b = "high entropy alloy high entropy alloy high entropy alloy"   # o mesmo conteúdo, 3x mais longo
c = "corrosion behaviour of stainless steel"                     # outro assunto

M = CountVectorizer().fit_transform([a, b, c]).toarray()
rot = ['a (curto)', 'b (a repetido 3x)', 'c (outro tema)']

print('DISTÂNCIA EUCLIDIANA (0 = idêntico, quanto maior mais diferente)')
print(pd.DataFrame(euclidean_distances(M).round(2), index=rot, columns=rot))
print()
print('SIMILARIDADE DE COSSENOS (1 = mesma direção, 0 = nada em comum)')
print(pd.DataFrame(cosine_similarity(M).round(2), index=rot, columns=rot))

O resultado é didaticamente perfeito. Pela **distância euclidiana**, `a` está *mais perto* de `c` (que fala de outro assunto!) do que de `b` — só porque `c` tem comprimento parecido com o de `a`. Já a **similaridade de cossenos** dá 1,00 entre `a` e `b` (mesma direção, apesar do comprimento triplo) e 0,00 entre ambos e `c` (nenhuma palavra em comum). O cosseno acertou; a euclidiana errou feio.

### Três formas de calcular

Vamos calcular a mesma coisa de três maneiras, da mais explícita à mais prática:

1. **NumPy na mão** — para entender o que a fórmula faz.
2. **SciPy** (`spatial.distance.cosine`) — devolve a *distância*, então a similaridade é `1 - distância`.
3. **scikit-learn** (`cosine_similarity`) — calcula todos os pares de uma vez e aceita matrizes esparsas diretamente. **É o que se usa na prática.**

In [ ]:
# 1) Implementação direta da fórmula com NumPy.
def cos_sim(a, b):
    """Similaridade de cossenos entre dois vetores NumPy 1-D."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Recriamos o BOW binário do mini-corpus.
vectorizer = CountVectorizer(tokenizer=spacy_tokenizer, token_pattern=None,
                             lowercase=False, binary=True)
bow_bin = vectorizer.fit_transform(mini_corpus)

v0 = bow_bin[0].toarray().ravel()   # ravel() achata (1, V) para (V,)
v1 = bow_bin[1].toarray().ravel()

print(f'NumPy   : {cos_sim(v0, v1):.4f}')

# 2) SciPy devolve DISTÂNCIA de cossenos; similaridade = 1 - distância.
print(f'SciPy   : {1 - spatial.distance.cosine(v0, v1):.4f}')

# 3) scikit-learn, aceitando matriz esparsa sem conversão.
print(f'sklearn : {cosine_similarity(bow_bin[0], bow_bin[1])[0][0]:.4f}')

In [ ]:
# A grande vantagem do sklearn: a matriz completa de todos os pares, de uma só vez.
sim = cosine_similarity(bow_bin)

tabela_sim = pd.DataFrame(sim.round(3),
                          index=[f'doc{k}' for k in range(len(mini_corpus))],
                          columns=[f'doc{k}' for k in range(len(mini_corpus))])

for k, t in enumerate(mini_corpus):
    print(f'doc{k}: {t[:70]}')
print()
tabela_sim

Como ler essa matriz de similaridade:

- A **diagonal é sempre 1,0** — todo documento é idêntico a si mesmo.
- A matriz é **simétrica**: sim(a,b) = sim(b,a). Metade dela é redundante.

### E agora o resultado surpreendente

Olhe a linha `doc0`: ela tem **0,000** contra `doc2`, `doc3` e `doc4`. O modelo está afirmando que "Research Progress on Refractory High Entropy Alloys" **não tem uma única palavra em comum** com "Discovery and design of fatigue-resistant high-entropy alloys". Isso é obviamente falso para qualquer leitor humano.

A causa é o `lowercase=False` que acabamos de ativar. O doc0 escreve `High Entropy Alloys` em caixa alta (é um título estilo *Title Case*); os docs 2, 3 e 4 escrevem `high entropy alloy` em minúsculas. Como são colunas **diferentes** do vocabulário, o produto escalar entre esses vetores é literalmente zero. O doc1, que também usa `High-Entropy` capitalizado, é o único que consegue similaridade não nula com o doc0 (0,228).

**Esta é a lição mais importante da seção.** Um parâmetro que parecia um detalhe inofensivo — preservar a caixa para não estragar fórmulas químicas — destruiu completamente a capacidade do modelo de reconhecer documentos do mesmo tema. Em vetorização, o pré-processamento **não é uma etapa preliminar**: ele *é* o modelo. Antes de culpar o algoritmo por um resultado ruim, inspecione o vocabulário.

Na prática, quem trabalha com corpora técnicos costuma resolver isso com um meio-termo: aplicar *lowercase* normalmente e proteger as fórmulas químicas por outro caminho (uma lista de exceções, um `EntityRuler` do spaCy, ou um casamento por expressão regular antes da normalização). Nas próximas seções voltaremos a usar minúsculas, via `tok_lema`.

### O outro defeito, o estrutural

Corrigida a caixa, restaria ainda um problema mais profundo: **todos** os documentos do corpus compartilham `high`, `entropy` e `alloy`, e essas palavras — presentes em toda parte — inflam artificialmente a similaridade entre quaisquer dois documentos. Num corpus temático como este, **BOW por contagem superestima sistematicamente a semelhança**. Mediremos exatamente esse efeito na Seção 3.5, e é o que o TF-IDF vem corrigir.

---
## 2.8 *n*-gramas: devolvendo um pouco de ordem ao saco

O BOW joga a ordem fora. Uma correção parcial e barata é tratar **sequências contíguas de *n* palavras** como se fossem termos únicos:

| *n* | Nome | Exemplo |
|---|---|---|
| 1 | unigrama | `high`, `entropy`, `alloy` |
| 2 | bigrama | `high entropy`, `entropy alloy` |
| 3 | trigrama | `high entropy alloy` |

A vantagem em texto técnico é grande, porque termos compostos têm significado próprio: `solid solution`, `grain boundary`, `phase transformation`, `wear resistance`. Como unigramas isolados, `solid` e `solution` dizem muito menos do que `solid solution`.

Bigramas também capturam parcialmente a **negação** — `not resistant` vira um termo próprio, distinto de `resistant`.

No `CountVectorizer` isso é controlado por `ngram_range=(min, max)`. O padrão é `(1, 1)`: só unigramas.

In [ ]:
%%time
# Efeito do ngram_range sobre o tamanho do vocabulário (corpus completo).
faixas = [(1, 1), (1, 2), (2, 2), (1, 3)]
linhas = []

for faixa in faixas:
    v = CountVectorizer(tokenizer=tok_lema, token_pattern=None,
                        lowercase=False, ngram_range=faixa)
    m = v.fit_transform(corpus)
    linhas.append({'ngram_range': str(faixa), 'nº de features': m.shape[1]})

pd.DataFrame(linhas)

**A explosão combinatória é o ponto principal aqui.** Saindo de unigramas para uni+bigramas, o vocabulário multiplica por um fator grande; incluir trigramas piora ainda mais. E a maior parte desses bigramas aparece em **um único documento** — não ajudam a generalizar nada, só inflam a matriz e o tempo de treino.

O antídoto são os parâmetros de corte de frequência de documento:

- **`min_df`** — descarta termos que aparecem em menos de *n* documentos (ou em menos de uma fração, se você passar um float). Elimina o ruído da cauda longa: erros de digitação, códigos únicos, bigramas acidentais.
- **`max_df`** — descarta termos que aparecem em mais de uma fração dos documentos. Funciona como uma lista de *stop words* automática e **específica do corpus**: em nosso caso, remove `high`, `entropy` e `alloy`, que estão em praticamente todo documento.

Vamos ver o efeito.

In [ ]:
%%time
# Uni+bigramas com corte de frequência.
v = CountVectorizer(tokenizer=tok_lema, token_pattern=None, lowercase=False,
                    ngram_range=(1, 2),
                    min_df=5,      # o termo tem que aparecer em pelo menos 5 documentos
                    max_df=0.5)    # e em no máximo 50% deles
m = v.fit_transform(corpus)

print(f'Vocabulário após min_df=5 e max_df=0.5: {m.shape[1]} features')
print()

# Alguns bigramas que sobreviveram ao corte.
# A lista sai em ordem alfabética, então tomamos uma amostra ESPAÇADA
# para ver termos de todo o alfabeto, e não só os que começam com 'a'.
bigramas = [t for t in v.get_feature_names_out() if ' ' in t]
print(f'Bigramas retidos: {len(bigramas)}')
print()
print(bigramas[::len(bigramas) // 30])

Olhe a amostra acima: `ab initio`, `anneal temperature`, `boundary strengthening`, `cryogenic temperature`, `fcc crystal`, `plastic strain`, `steel substrate`... São **conceitos** da área, não pares acidentais de palavras. Isso é informação que unigramas isolados simplesmente não conseguem representar — e explica por que uni+bigramas com `min_df` costuma ser a configuração padrão de quem trabalha com corpora técnicos.

Nem tudo é ouro, claro: a lista também traz colagens sem sentido próprio, como `effect high`, `investigate find` ou `metal high`, que são resíduos da remoção de *stop words* (o bigrama original era algo como *"effect of high"*). Elas ocupam espaço, mas em geral são inofensivas — recebem IDF alto demais para pesar e, na prática, se diluem.

---
## 2.9 Exercícios — Bag-of-Words

Faça em duplas; discutiremos as respostas em aula.

In [ ]:
# ------------------------------------------------------------------
# EXERCÍCIO 1
# Escreva um tokenizador `meu_tokenizador(doc)` que:
#   (a) use o spaCy;
#   (b) devolva o LEMA em minúsculas;
#   (c) descarte pontuação, espaços e stop words;
#   (d) mantenha APENAS substantivos e adjetivos (dica: o atributo t.pos_
#       assume os valores 'NOUN', 'PROPN', 'ADJ', 'VERB', ...).
# A hipótese a testar: em texto científico, os conceitos estão concentrados
# nos sintagmas nominais. Verbos como "show", "increase", "study" são
# fraseologia de artigo, não conteúdo.
#
# Depois: monte o BOW do corpus completo com ele e compare o tamanho do
# vocabulário com o da tabela da Seção 2.6.
# ATENÇÃO: para usar t.pos_ o componente 'tagger' precisa estar ATIVO.
# ------------------------------------------------------------------

def meu_tokenizador(doc):
    pass

In [ ]:
# ------------------------------------------------------------------
# EXERCÍCIO 2
# Usando o BOW BINÁRIO do mini-corpus (Seção 2.7), responda:
#   (a) Qual par de documentos é o MAIS parecido? E o MENOS?
#   (b) Olhando os títulos, o resultado faz sentido tematicamente?
#   (c) Refaça a matriz de similaridade usando binary=False (contagens).
#       Os valores mudam? A ORDENAÇÃO dos pares muda? Por quê?
#   (d) Agora refaça removendo as stop words e lematizando (use tok_lema).
#       As similaridades sobem ou descem? Explique o que aconteceu.
# ------------------------------------------------------------------

In [ ]:
# ------------------------------------------------------------------
# EXERCÍCIO 3 — o problema das palavras fora do vocabulário (OOV)
#
# O texto abaixo é um resumo REAL, mas de um artigo que NÃO está no nosso
# corpus (foi inventado com vocabulário parcialmente novo).
# Note que usamos transform() e NÃO fit_transform(): o vocabulário já foi
# aprendido e não queremos reaprendê-lo — queremos justamente ver o que
# acontece com as palavras que ele não conhece.
#
# TAREFA:
#   (a) Vetorize o texto com o `vectorizer` já ajustado.
#   (b) Quantos tokens do texto sobreviveram (quantas células não nulas)?
#   (c) Liste as palavras do texto que NÃO estão no vocabulário.
#       Dica: compare o conjunto de tokens do texto com vectorizer.vocabulary_
#   (d) Calcule a similaridade dele com cada documento do mini-corpus.
#       O resultado ainda é útil, mesmo com informação perdida?
# ------------------------------------------------------------------

texto_novo = ["Biocompatible titanium-niobium-zirconium alloys were fabricated by "
              "spark plasma sintering and evaluated for orthopaedic implants. "
              "The samples exhibited low elastic modulus and excellent corrosion "
              "resistance in simulated body fluid."]

# novo_bow = vectorizer.transform(texto_novo)

---
# 3. TF-IDF

## 3.1 A motivação, medida no nosso corpus

Já suspeitamos do problema. Vamos comprová-lo: quais são os termos mais frequentes nos 1.000 resumos?

In [ ]:
%%time
# Contagem bruta sobre o corpus completo, com o tokenizador de lemas.
cv_full = CountVectorizer(tokenizer=tok_lema, token_pattern=None, lowercase=False)
contagens = cv_full.fit_transform(corpus)

vocab_full = cv_full.get_feature_names_out()
freq_total = np.asarray(contagens.sum(axis=0)).ravel()   # soma por coluna = total no corpus
ordem = np.argsort(freq_total)[::-1]

print(f'Vocabulário: {len(vocab_full)} termos')
print()
print('OS 20 TERMOS MAIS FREQUENTES DO CORPUS')
print('-' * 45)
for j in ordem[:20]:
    print(f'{vocab_full[j]:<18} {freq_total[j]:>6}')

Aí está o problema, em números. Os termos no topo são `alloy`, `high`, `entropy`, `phase`, `hea` — exatamente o assunto de **todos** os 1.000 artigos. Eles dominam os vetores e, portanto, dominam qualquer cálculo de similaridade, sem contribuir com **nenhuma** capacidade de distinguir um documento do outro.

Note que **remover *stop words* não resolveu**: `alloy` e `entropy` não são *stop words* do inglês, são palavras de conteúdo pleno. O que as torna inúteis aqui não é a língua — é o **corpus**. Numa coleção de notícias, `entropy` seria um termo raríssimo e altamente informativo.

Precisamos, então, de um peso que seja **relativo ao corpus**: alto para termos que aparecem em poucos documentos, baixo para os que aparecem em todos. Esse peso é o **IDF**.

## 3.2 A fórmula

TF-IDF é o produto de dois fatores, calculado para cada par (termo *t*, documento *d*):

### TF — *Term Frequency*

Quão importante é *t* **dentro de** *d*. Na formulação do scikit-learn, é simplesmente a contagem bruta:

$$\text{tf}(t, d) = \text{nº de ocorrências de } t \text{ em } d$$

(Existem variantes: frequência relativa, log-normalizada $1 + \log(\text{tf})$, normalização pelo termo mais frequente. Todas atacam o mesmo ponto: repetir 20 vezes não deveria valer 20 vezes mais do que repetir 1.)

### IDF — *Inverse Document Frequency*

Quão **raro** é *t* no corpus inteiro. Começando pela **frequência de documento** df(*t*) — em quantos documentos *t* aparece, ao menos uma vez — a forma clássica é:

$$\text{idf}(t) = \log\frac{N}{\text{df}(t)}$$

O scikit-learn usa por padrão a variante **suavizada** (`smooth_idf=True`), que soma 1 ao numerador, ao denominador e ao resultado:

$$\text{idf}(t) = \log\frac{1 + N}{1 + \text{df}(t)} + 1$$

Por que os três "+1"?

- **`+1` no denominador** evita divisão por zero para termos que existem no vocabulário mas têm df = 0 (acontece quando você aplica um vocabulário aprendido em outro corpus). É como se houvesse um documento fantasma contendo todos os termos.
- **`+1` no numerador** mantém o argumento do logaritmo coerente com essa suavização.
- **`+1` no resultado** garante idf ≥ 1, de modo que um termo presente em *todos* os documentos receba peso 1 e não seja **zerado**. Sem ele, log(N/N) = 0 apagaria completamente esses termos.

### O produto

$$\text{tfidf}(t, d) = \text{tf}(t, d) \times \text{idf}(t)$$

E, por fim, o scikit-learn **normaliza cada linha** pela norma L2 (`norm='l2'`), fazendo todo vetor de documento ter comprimento 1. Isso tem uma consequência prática elegante: para vetores unitários, a similaridade de cossenos se reduz ao **produto escalar** puro, o que torna a busca mais rápida.

O comportamento resultante é o desejado:

| Situação | tf | idf | tf-idf |
|---|---|---|---|
| termo frequente no doc **e** raro no corpus | alto | alto | **muito alto** ← o termo ideal |
| termo frequente no doc e comum no corpus | alto | ~1 | médio |
| termo raro no doc e raro no corpus | baixo | alto | médio |
| termo em todos os documentos | qualquer | 1 | baixo |

### Verificando a fórmula numericamente

Nada de acreditar na fórmula por fé. Vamos calcular o IDF na mão para o mini-corpus e comparar com o que o scikit-learn produz.

In [ ]:
# IDF calculado à mão vs. IDF do scikit-learn, no mini-corpus.
cv_mini = CountVectorizer()
bow_mini = cv_mini.fit_transform(mini_corpus)

tv_mini = TfidfVectorizer()
tv_mini.fit(mini_corpus)

N = len(mini_corpus)
df_t = np.asarray((bow_mini > 0).sum(axis=0)).ravel()   # em quantos docs cada termo aparece
idf_manual = np.log((1 + N) / (1 + df_t)) + 1           # fórmula suavizada do sklearn

print('As duas contas batem?', np.allclose(idf_manual, tv_mini.idf_))
print()

vocab_mini = list(tv_mini.get_feature_names_out())
print(f'{"termo":<14}{"df":>4}{"idf":>9}')
print('-' * 27)
for termo in ['high', 'entropy', 'alloy', 'research', 'corrosion', 'hydrogen']:
    j = vocab_mini.index(termo)
    print(f'{termo:<14}{df_t[j]:>4}{tv_mini.idf_[j]:>9.3f}')

Leia a tabela de baixo para cima: `hydrogen` aparece em 1 dos 5 documentos e recebe o **maior** peso; `high` e `entropy` aparecem nos 5 e recebem o peso **mínimo** (1,0). É exatamente o comportamento que queríamos — sem escrever uma linha de lógica específica do domínio. **O IDF descobre sozinho, a partir da estatística do corpus, quais palavras não discriminam nada.**

## 3.3 `TfidfVectorizer` sobre os 1.000 resumos

A API é idêntica à do `CountVectorizer` — o que muda é o valor guardado em cada célula. Vamos usar o tokenizador de lemas sem *stop words*, que foi o que se mostrou mais adequado na Seção 2.6.

- Documentação: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

> Curiosidade de implementação: `TfidfVectorizer` é literalmente um `CountVectorizer` encadeado com um `TfidfTransformer`. Se você já tem a matriz de contagens, pode aplicar só o transformador e economizar a tokenização.

In [ ]:
%%time
vectorizer_tfidf = TfidfVectorizer(
    tokenizer=tok_lema,
    token_pattern=None,
    lowercase=False,
    # smooth_idf=True e norm='l2' são os padrões; explicitados aqui para deixar visível.
    smooth_idf=True,
    norm='l2'
)

X = vectorizer_tfidf.fit_transform(corpus)
termos_tfidf = vectorizer_tfidf.get_feature_names_out()

In [ ]:
print(f'Formato da matriz : {X.shape}   (documentos × termos)')
print(f'Vocabulário       : {len(termos_tfidf)} termos')
print(f'Células totais    : {X.shape[0] * X.shape[1]:,}')
print(f'Células não nulas : {X.nnz:,}')
print(f'Densidade         : {100 * X.nnz / (X.shape[0] * X.shape[1]):.2f}%')
print()

# Confirmando a normalização L2: toda linha deve ter comprimento 1.
normas = np.sqrt(np.asarray(X.multiply(X).sum(axis=1))).ravel()
print(f'Norma L2 das linhas — mín: {normas.min():.4f}, máx: {normas.max():.4f}')

In [ ]:
# O documento 0 em formato esparso: agora com pesos reais em vez de contagens inteiras.
print(X[0])

## 3.4 Diagnóstico: o que o TF-IDF aprendeu

Duas inspeções que **todo mundo deveria fazer** depois de treinar um vetorizador — e que quase ninguém faz.

### (a) Os termos de maior peso em um documento

Se o TF-IDF está funcionando, os termos de maior peso em cada documento devem parecer um **resumo automático por palavras-chave** dele.

In [ ]:
def termos_principais(i, k=12):
    """Devolve os k termos de maior peso TF-IDF no documento i."""
    linha = X[i].toarray().ravel()
    top = np.argsort(linha)[::-1][:k]
    return pd.DataFrame({
        'termo': [termos_tfidf[j] for j in top],
        'peso' : [round(float(linha[j]), 4) for j in top],
    })

i = 53
print('TÍTULO  :', df['Article Title'][i])
print('KEYWORDS:', df['Author Keywords'][i])
print()
termos_principais(i)

Compare a coluna `termo` com a linha `KEYWORDS` acima dela — que foram escritas **por seres humanos**, os próprios autores do artigo. A sobreposição costuma ser notável, e ninguém programou nada específico do domínio: o TF-IDF extraiu as palavras-chave apenas contando e reponderando.

É esse fenômeno que faz o TF-IDF continuar em uso, décadas depois de inventado, como *baseline* de extração de palavras-chave.

Rode a célula abaixo para outros documentos e observe se o padrão se mantém.

In [ ]:
for i in [0, 7, 120, 500]:
    linha = X[i].toarray().ravel()
    top = np.argsort(linha)[::-1][:8]
    print(f'[{i}] {df["Article Title"][i][:78]}')
    print('     ', ', '.join(termos_tfidf[j] for j in top))
    print()

### (b) Os extremos do IDF

O vetor `idf_` guarda o peso de raridade de cada termo. Olhar suas duas pontas revela o "vocabulário de fundo" do corpus (embaixo) e sua cauda longa (em cima).

In [ ]:
idf = vectorizer_tfidf.idf_
ordem_idf = np.argsort(idf)

print('MENOR IDF — o "vocabulário de fundo": está em quase todo documento,')
print('             quase não ajuda a distinguir nada')
print('-' * 62)
for j in ordem_idf[:15]:
    print(f'{termos_tfidf[j]:<18} idf = {idf[j]:.3f}')

print()
print('MAIOR IDF — a cauda longa: aparece em 1 ou 2 documentos.')
print('             Altamente específico, mas também onde mora o ruído')
print('-' * 62)
for j in ordem_idf[-15:]:
    print(f'{termos_tfidf[j]:<18} idf = {idf[j]:.3f}')

Os termos de **menor** IDF (`alloy`, `high`, `entropy`, `phase`, `property`) formam, na prática, uma **lista de *stop words* descoberta automaticamente para este corpus** — sem dicionário, sem intervenção humana. Esse é um resultado bonito e vale destacá-lo em aula.

Os de **maior** IDF são um caso mais ambíguo. Uma fórmula química rara é ouro para busca; mas erros de digitação e artefatos de conversão de PDF também moram aí — e recebem peso máximo, o que os torna capazes de distorcer similaridades. Por isso, na prática, corta-se a cauda com `min_df`.

In [ ]:
%%time
# Cortando as duas pontas da distribuição.
vectorizer_cortado = TfidfVectorizer(
    tokenizer=tok_lema, token_pattern=None, lowercase=False,
    min_df=3,        # o termo tem que estar em pelo menos 3 documentos
    max_df=0.5       # e em no máximo metade deles
)
Xc = vectorizer_cortado.fit_transform(corpus)

print(f'Sem corte : {X.shape[1]} termos')
print(f'Com corte : {Xc.shape[1]} termos '
      f'({100 * (1 - Xc.shape[1] / X.shape[1]):.0f}% do vocabulário eliminado)')

Mais da metade do vocabulário desapareceu — e, na maioria das tarefas, **sem perda de qualidade**, porque o que saiu era ruído da cauda longa ou vocabulário de fundo. Menos colunas significa menos memória, treino mais rápido e menor risco de sobreajuste.

Aqui, porém, cabe um alerta pedagógico honesto: `min_df` também elimina fórmulas químicas raras, e num corpus de materiais essas fórmulas podem ser justamente o que o usuário procura. Se o objetivo for buscar por `TiZrHfScMo`, cortar termos raros é um tiro no pé. **Toda escolha de pré-processamento é uma aposta sobre o uso futuro** — por isso mantemos as duas matrizes, `X` (completa) e `Xc` (cortada), e usaremos `X` na busca.

---
## 3.5 Construindo um motor de busca

Temos tudo o que é preciso para **recuperação de informação**. O algoritmo tem três passos:

1. **Vetorizar a consulta** com o *mesmo* vetorizador — usando `transform`, jamais `fit_transform`. A consulta precisa viver no mesmo espaço vetorial que os documentos, com as mesmas colunas na mesma ordem.
2. **Calcular a similaridade de cossenos** entre a consulta e cada um dos 1.000 documentos.
3. **Ordenar** em ordem decrescente e devolver os *k* primeiros.

É o esqueleto de qualquer sistema de busca lexical — inclusive dos comerciais, que acrescentam índice invertido, BM25 e centenas de sinais de ranqueamento por cima disso.

In [ ]:
# Passo 1: a consulta vira um vetor no MESMO espaço dos documentos.
consulta = ["hydrogen storage in refractory high entropy alloys"]
v_consulta = vectorizer_tfidf.transform(consulta)

print(f'Formato do vetor de consulta: {v_consulta.shape}')
print(f'Termos reconhecidos (não nulos): {v_consulta.nnz}')
print()
print('Como a consulta ficou representada:')
linha_q = v_consulta.toarray().ravel()
for j in np.argsort(linha_q)[::-1][:linha_q.nonzero()[0].size]:
    print(f'  {termos_tfidf[j]:<16} {linha_q[j]:.4f}')

Repare que os termos da consulta **não** têm todos o mesmo peso. `storage`, `hydrogen` e `refractory` pesam de três a cinco vezes mais do que `high`, `entropy` e `alloy` — porque o IDF já sabe que estes últimos estão em todo lugar. A consulta, sozinha, "entende" que o que o usuário realmente quer é *armazenamento de hidrogênio* em ligas *refratárias*, e que a expressão "high entropy alloys" é só o pano de fundo. Perceba também que a *stop word* `in` foi descartada pelo tokenizador, sobrando 6 termos dos 7 digitados.

In [ ]:
# Passo 2: similaridade da consulta contra TODOS os documentos.
# flatten() porque cosine_similarity devolve matriz (N, 1) e queremos um vetor (N,).
similaridades = cosine_similarity(X, v_consulta).flatten()

print(f'Vetor de similaridades: {similaridades.shape}')
print(f'Maior : {similaridades.max():.4f}')
print(f'Média : {similaridades.mean():.4f}')
print(f'Documentos com similaridade zero: {(similaridades == 0).sum()}')

In [ ]:
# Passo 3: os índices dos k maiores valores.
def top_k(arr, k):
    """Índices dos k maiores elementos de arr, em ordem decrescente.

    np.argsort ordena de forma CRESCENTE e devolve ÍNDICES.
    Os maiores ficam no fim, então percorremos de trás para frente
    com o passo -1 e paramos no k-ésimo.
    (Para arrays muito grandes, np.argpartition é mais rápido.)
    """
    kth = (k + 1) * -1
    return np.argsort(arr)[:kth:-1]

indices = top_k(similaridades, 5)
print('Índices dos 5 mais relevantes:', indices)
print('Similaridades              :', similaridades[indices].round(4))

### Empacotando em uma função de busca

Índices e números soltos não convencem ninguém. Vamos montar uma função que devolve os resultados de forma legível — título, periódico, ano e pontuação — como faria um sistema real.

In [ ]:
def buscar(texto_consulta, k=5, vetorizador=None, matriz=None, mostrar_resumo=False):
    """Busca os k artigos mais similares à consulta, por similaridade de cossenos.

    Parâmetros
    ----------
    texto_consulta : str   — a consulta em linguagem natural
    k              : int   — quantos resultados devolver
    vetorizador    : o TfidfVectorizer já ajustado (fit)
    matriz         : a matriz TF-IDF do corpus, gerada pelo mesmo vetorizador
    mostrar_resumo : bool  — se True, imprime o início do abstract
    """
    vetorizador = vetorizador if vetorizador is not None else vectorizer_tfidf
    matriz = matriz if matriz is not None else X

    q = vetorizador.transform([texto_consulta])       # transform, NUNCA fit_transform
    sims = cosine_similarity(matriz, q).flatten()
    idx = top_k(sims, k)

    print(f'CONSULTA: "{texto_consulta}"')
    print('=' * 100)
    for posicao, i in enumerate(idx, start=1):
        print(f'{posicao}. [score {sims[i]:.4f}]  {df["Article Title"][i]}')
        print(f'   {df["Source Title"][i]}, {int(df["Publication Year"][i])}')
        if mostrar_resumo:
            print(f'   {df["Abstract"][i][:260]}...')
        print()
    return idx

In [ ]:
_ = buscar("hydrogen storage in refractory high entropy alloys", k=5)

Funciona — e funciona bem. Os títulos recuperados são, todos, sobre armazenamento de hidrogênio em ligas de alta entropia. Sem modelo treinado, sem rede neural, sem GPU: apenas contagens de palavras reponderadas por raridade e um cosseno.

Vamos testar outras consultas, de temas variados.

In [ ]:
_ = buscar("corrosion resistance of coatings produced by laser cladding", k=5)

In [ ]:
_ = buscar("mechanical behaviour at cryogenic temperature and twinning", k=5)

In [ ]:
_ = buscar("first principles DFT calculation of lattice distortion", k=5, mostrar_resumo=True)

### BOW puro vs. TF-IDF: a comparação decisiva

Vale a pena provar que o IDF faz diferença de fato, e não só na teoria. Vamos rodar **a mesma consulta** sobre a matriz de contagens brutas e sobre a matriz TF-IDF, lado a lado.

In [ ]:
consulta_teste = "wear resistance and hardness of coatings"

# --- Ranking com BOW de contagens brutas ---
q_bow = cv_full.transform([consulta_teste])
sims_bow = cosine_similarity(contagens, q_bow).flatten()
idx_bow = top_k(sims_bow, 5)

# --- Ranking com TF-IDF ---
q_tfidf = vectorizer_tfidf.transform([consulta_teste])
sims_tfidf = cosine_similarity(X, q_tfidf).flatten()
idx_tfidf = top_k(sims_tfidf, 5)

print(f'CONSULTA: "{consulta_teste}"\n')
print('--- BOW (contagens brutas) ---')
for p, i in enumerate(idx_bow, 1):
    print(f'{p}. [{sims_bow[i]:.3f}] {df["Article Title"][i][:82]}')
print()
print('--- TF-IDF ---')
for p, i in enumerate(idx_tfidf, 1):
    print(f'{p}. [{sims_tfidf[i]:.3f}] {df["Article Title"][i][:82]}')
print()
print(f'Documentos em comum no top-5: {len(set(idx_bow) & set(idx_tfidf))}')
print(f'Contraste do ranking (melhor score ÷ score médio):')
print(f'  BOW    : {sims_bow.max() / sims_bow.mean():.1f}x')
print(f'  TF-IDF : {sims_tfidf.max() / sims_tfidf.mean():.1f}x')

As diferenças a discutir em aula:

1. **Os rankings divergem**, mesmo com boa parte dos documentos coincidindo. Sob TF-IDF, o artigo sobre atrito e desgaste em alta temperatura sobe para a primeira posição, e entra no top-5 um artigo sobre revestimentos por aspersão térmica que o BOW não havia recuperado. O BOW distribui crédito para documentos que apenas repetem muito `alloy` e `high`; o TF-IDF concentra o crédito em `wear`, `hardness` e `coating` — os termos raros e informativos da consulta.
2. **O TF-IDF separa melhor o joio do trigo.** É o que mede a última linha: a razão entre a melhor pontuação e a pontuação média do corpus. Quanto maior essa razão, mais destacado fica o resultado relevante em relação ao ruído de fundo — e o TF-IDF vence com folga.

### Medindo a inflação de similaridade do BOW

Fica ainda a afirmação que fizemos na Seção 2.7: a de que o BOW **superestima** a semelhança entre documentos, porque todos compartilham o vocabulário de fundo. Vamos medir isso diretamente, comparando a similaridade média entre pares de documentos *quaisquer* do corpus nas duas representações.

In [ ]:
# Amostra de 200 documentos (a matriz completa de 1000x1000 pares é grande
# e desnecessária para o argumento).
amostra = np.random.default_rng(0).choice(len(corpus), size=200, replace=False)

pares_bow   = cosine_similarity(contagens[amostra])
pares_tfidf = cosine_similarity(X[amostra])

fora_diagonal = ~np.eye(len(amostra), dtype=bool)   # ignora a diagonal (doc consigo mesmo)

print('Similaridade média entre DOIS DOCUMENTOS QUAISQUER do corpus:')
print(f'  BOW (contagens) : {pares_bow[fora_diagonal].mean():.3f}')
print(f'  TF-IDF          : {pares_tfidf[fora_diagonal].mean():.3f}')

O contraste é grande: pelo BOW, dois artigos escolhidos ao acaso já parecem consideravelmente semelhantes — mas essa semelhança é quase toda fabricada pelas palavras que estão em todo lugar. O TF-IDF derruba esse piso para perto de zero, que é onde ele deve estar: **dois artigos aleatórios do corpus não têm, de fato, quase nada em comum além do tema geral.**

E é justamente isso que torna o TF-IDF melhor para busca. Não é que ele encontre documentos "mais parecidos" — é que ele reserva as pontuações altas para os casos em que a semelhança é **real**.

## 3.6 Recomendação: "artigos parecidos com este"

A mesma máquina resolve outro problema, sem nenhuma mudança de código: em vez de comparar o corpus com uma consulta digitada, comparamos o corpus com **um documento que já está nele**. É a funcionalidade "artigos relacionados" de qualquer portal acadêmico.

O único cuidado é excluir o próprio documento do ranking — ele teria similaridade 1,0 consigo mesmo e ocuparia sempre a primeira posição.

In [ ]:
def artigos_relacionados(i, k=5):
    """Os k artigos mais similares ao artigo de índice i (excluindo ele próprio)."""
    sims = cosine_similarity(X[i], X).flatten()
    sims[i] = -1                     # remove o próprio documento do páreo
    idx = top_k(sims, k)

    print('ARTIGO DE REFERÊNCIA')
    print(f'  {df["Article Title"][i]}')
    print(f'  {df["Source Title"][i]}, {int(df["Publication Year"][i])}')
    print()
    print('ARTIGOS RELACIONADOS')
    for posicao, j in enumerate(idx, start=1):
        print(f'  {posicao}. [{sims[j]:.4f}] {df["Article Title"][j]}')
    return idx

_ = artigos_relacionados(53, k=5)

In [ ]:
_ = artigos_relacionados(7, k=5)

---
## 3.7 Bônus: agrupando o corpus sem supervisão

Um último uso, para mostrar que os vetores servem a bem mais do que busca. Como cada documento agora é um ponto num espaço de milhares de dimensões, podemos aplicar qualquer algoritmo de agrupamento. O **k-means** particiona os 1.000 artigos em *k* grupos por proximidade, sem nunca ter visto um rótulo.

Depois, olhando os termos de maior peso no **centroide** de cada grupo, conseguimos *nomear* o subtema que o algoritmo descobriu sozinho. Isso se chama **modelagem exploratória de tópicos**, e é uma forma barata e eficaz de mapear a estrutura de uma literatura científica.

Usaremos a matriz `Xc` (com corte de frequência): para agrupar, o vocabulário de fundo é ruído puro.

In [ ]:
%%time
from sklearn.cluster import KMeans

K = 6
km = KMeans(n_clusters=K, n_init=10, random_state=42)
grupos = km.fit_predict(Xc)

termos_c = vectorizer_cortado.get_feature_names_out()

print(f'{K} grupos descobertos automaticamente nos {len(corpus)} resumos\n')
for c in range(K):
    centro = km.cluster_centers_[c]
    principais = np.argsort(centro)[::-1][:10]
    n = int((grupos == c).sum())
    print(f'GRUPO {c}  ({n} artigos)')
    print('  termos:', ', '.join(termos_c[j] for j in principais))
    exemplo = df['Article Title'][grupos == c].iloc[0]
    print(f'  exemplo: {exemplo[:88]}')
    print()

Os grupos costumam sair bastante interpretáveis — tipicamente separando propriedades mecânicas e tração; revestimentos, desgaste e corrosão; deformação, discordâncias e maclação; filmes finos e oxidação; e um grupo mais genérico de projeto e caracterização de ligas.

**Nenhum rótulo foi fornecido.** Toda essa estrutura veio das estatísticas de coocorrência de palavras, transformadas em geometria. É uma boa nota para fechar a aula: vetorização é o que converte linguagem em algo sobre o qual a matemática tem o que dizer.

In [ ]:
# Como os grupos se distribuem ao longo do tempo — dá para ver quais
# subtemas cresceram nos últimos anos do corpus.
tabela = pd.crosstab(df['Publication Year'].astype('Int64'), grupos)
tabela.columns = [f'grupo {c}' for c in tabela.columns]
tabela.tail(10)

---
# 4. Limitações e o caminho adiante

Construímos um motor de busca funcional com pouquíssimo código. Mas é importante ser claro sobre o que essa abordagem **não** consegue fazer.

### 4.1 O que o TF-IDF não resolve

| Limitação | O que acontece no nosso corpus |
|---|---|
| **Sinonímia** | uma consulta por `mechanical strength` não encontra um artigo que só diz `yield stress`. São colunas diferentes e ortogonais: a similaridade é exatamente zero. |
| **Polissemia** | `phase` significa fase cristalográfica aqui, mas fase de um processo em outro contexto. Uma só coluna para os dois sentidos. |
| **Vocabulário fixo (OOV)** | uma consulta com um termo que não estava no corpus de treino é simplesmente **ignorada**, sem aviso. |
| **Ordem e negação** | `not corrosion resistant` e `corrosion resistant` têm vetores quase idênticos. |
| **Dimensionalidade** | milhares de colunas, 99% zeros. Cresce com o corpus e não comprime nada. |
| **Zero conhecimento de mundo** | o modelo não sabe que `Fe` é ferro, nem que `BCC` e `FCC` são estruturas cristalinas alternativas. |

### 4.2 Melhorias ainda dentro do paradigma lexical

Antes de partir para redes neurais, várias melhorias clássicas cabem aqui:

- **Índice invertido.** Hoje varremos os 1.000 documentos a cada consulta. Um índice invertido (mapa `termo → documentos que o contêm`) permite visitar apenas os documentos que compartilham algum termo com a consulta. É o que Elasticsearch e Lucene fazem por baixo do pano. https://en.wikipedia.org/wiki/Inverted_index
- **Okapi BM25.** O padrão de fato em busca lexical moderna. Refina o TF-IDF com saturação do TF (a 20ª ocorrência de um termo vale pouquíssimo mais que a 19ª) e normalização por comprimento de documento. Costuma superar o TF-IDF em recuperação. https://en.wikipedia.org/wiki/Okapi_BM25
- **Avaliação quantitativa.** Nós julgamos os resultados "de olho". Um sistema sério mede **precisão**, **revocação**, F1, MAP e nDCG contra julgamentos de relevância. https://en.wikipedia.org/wiki/Precision_and_recall
- **Redução de dimensionalidade.** SVD truncado sobre a matriz TF-IDF (a *Latent Semantic Analysis*, LSA) comprime milhares de colunas em algumas centenas de dimensões densas, e captura alguma sinonímia de quebra. É a ponte histórica entre TF-IDF e *embeddings*.

### 4.3 Para onde vamos: representações densas

Todas as limitações da tabela 4.1 têm a mesma raiz: **cada palavra é uma dimensão isolada e ortogonal a todas as outras**. Não existe qualquer noção de que `strength` e `stress` são relacionadas — para o modelo, elas estão tão distantes quanto `strength` e `banana`.

A próxima aula ataca isso de frente com os **word embeddings** (word2vec, GloVe, fastText): em vez de um vetor esparso de 5.000 dimensões com um único 1, cada palavra vira um vetor **denso** de algumas centenas de dimensões, **aprendido** a partir dos contextos em que ela ocorre. Palavras que aparecem em contextos parecidos ganham vetores parecidos — e, de repente, `strength` e `stress` ficam próximas sem que ninguém tenha escrito uma regra.

Ainda assim: o TF-IDF continua sendo um *baseline* obrigatório. É rápido, interpretável, não precisa de treino e, em muitas tarefas de classificação de tópicos, **empata com métodos neurais bem mais caros**. Nunca comece um projeto sem medir o TF-IDF primeiro.

---
# 5. Exercícios — TF-IDF e recuperação de informação

Estes exercícios usam o mesmo `savedrecs1.xls`. Entregue um notebook com o código e uma discussão curta dos resultados.

In [ ]:
# ------------------------------------------------------------------
# EXERCÍCIO 4 — busca com n-gramas
#
# Construa um TfidfVectorizer com ngram_range=(1, 2), min_df=3 e max_df=0.5,
# usando tok_lema. Depois:
#   (a) Rode as MESMAS consultas da Seção 3.5 nas duas matrizes (unigramas
#       e uni+bigramas) e compare os top-5.
#   (b) Para a consulta "solid solution strengthening mechanism", o ranking
#       melhora com bigramas? Por que você esperaria isso?
#   (c) Qual o custo em número de features e em tempo de vetorização?
#       Vale a pena?
# ------------------------------------------------------------------

In [ ]:
# ------------------------------------------------------------------
# EXERCÍCIO 5 — indexar título + palavras-chave junto com o resumo
#
# Sistemas de busca reais indexam vários campos, geralmente com PESOS
# diferentes (o título "vale mais" que o corpo do texto).
#   (a) Crie uma coluna `texto_completo` concatenando Article Title,
#       Author Keywords e Abstract. Cuidado com valores nulos em
#       Author Keywords (use .fillna('')).
#   (b) Refaça a vetorização TF-IDF sobre esse campo combinado.
#   (c) Compare os resultados de busca com os da Seção 3.5. Melhoraram?
#   (d) DESAFIO: repita o título 3 vezes na concatenação — é o truque
#       clássico e grosseiro de dar peso a um campo. O que muda no ranking?
#       Por que isso funciona, dada a fórmula do TF?
# ------------------------------------------------------------------

In [ ]:
# ------------------------------------------------------------------
# EXERCÍCIO 6 — avaliação com um gabarito informal
#
# As Author Keywords são, de certa forma, um "gabarito" produzido por
# humanos: dizem do que o artigo trata.
#   (a) Escolha 5 consultas cujos temas você sabe existirem no corpus
#       (ex.: "corrosion", "hydrogen storage", "thin film", "wear",
#       "additive manufacturing").
#   (b) Para cada uma, recupere o top-10 e conte quantos dos artigos
#       recuperados têm o termo da consulta nas Author Keywords.
#       Essa é uma PRECISÃO@10 aproximada.
#   (c) Compare a precisão@10 do BOW com a do TF-IDF.
#   (d) Discuta: por que essa métrica SUBESTIMA o desempenho real do
#       sistema? (Dica: pense num artigo relevantíssimo cujas keywords
#       usem um sinônimo do termo da consulta.)
# ------------------------------------------------------------------

In [ ]:
# ------------------------------------------------------------------
# EXERCÍCIO 7 — evolução temporal do vocabulário
#
# Divida o corpus em dois períodos: até 2018 e de 2019 em diante.
#   (a) Ajuste um TfidfVectorizer SEPARADO para cada período.
#   (b) Liste os 20 termos de maior peso MÉDIO em cada um.
#   (c) Quais termos aparecem apenas no período recente? Eles indicam
#       tendências reais da área (ex.: manufatura aditiva, aprendizado
#       de máquina aplicado a materiais)?
#   (d) CUIDADO METODOLÓGICO: os dois vetorizadores têm vocabulários
#       DIFERENTES, então os pesos não são diretamente comparáveis.
#       Como você tornaria a comparação justa? (Dica: `vocabulary=`.)
# ------------------------------------------------------------------

In [ ]:
# ------------------------------------------------------------------
# EXERCÍCIO 8 (desafio) — implementar TF-IDF do zero
#
# Sem usar TfidfVectorizer, apenas com NumPy e o CountVectorizer:
#   (a) Obtenha a matriz de contagens do corpus.
#   (b) Calcule df(t) para cada termo (em quantos documentos ele aparece).
#   (c) Calcule idf(t) = log((1+N)/(1+df)) + 1.
#   (d) Multiplique cada coluna da matriz de contagens pelo seu idf.
#   (e) Normalize cada LINHA pela norma L2.
#   (f) Verifique com np.allclose que sua matriz é igual à do sklearn.
#
# Dica: matrizes esparsas suportam .multiply(). Para expandir o idf ao
# longo das colunas, veja scipy.sparse.diags.
# ------------------------------------------------------------------

---
# Referências

### Material do curso original
- NLP Demystified — https://nlpdemystified.org
- Notebooks originais — https://github.com/futuremojo/nlp-demystified
- Módulo *Basic Bag-of-Words* — https://www.nlpdemystified.org/course/basic-bag-of-words
- Módulo *TF-IDF* — https://www.nlpdemystified.org/course/tf-idf

### Documentação das bibliotecas
- `CountVectorizer` — https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html
- `TfidfVectorizer` — https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
- Extração de features de texto (guia do scikit-learn) — https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction
- `cosine_similarity` — https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html
- Matrizes esparsas CSR (SciPy) — https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_matrix.html
- spaCy: modelos e uso — https://spacy.io/usage/models

### Fundamentos teóricos
- Jurafsky & Martin, *Speech and Language Processing* (3ª ed., rascunho aberto), cap. 6 — *Vector Semantics and Embeddings* — https://web.stanford.edu/~jurafsky/slp3/
- Manning, Raghavan & Schütze, *Introduction to Information Retrieval*, caps. 6 e 7 — https://nlp.stanford.edu/IR-book/
- Índice invertido — https://en.wikipedia.org/wiki/Inverted_index
- Okapi BM25 — https://en.wikipedia.org/wiki/Okapi_BM25
- Precisão e revocação — https://en.wikipedia.org/wiki/Precision_and_recall

### Corpus
- `savedrecs1.xls` — exportação da Web of Science (Clarivate) com 1.000 registros sobre ligas de alta entropia. Uso restrito a fins didáticos nesta disciplina.